In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("./data/sekunder/tomato irrigation dataset.csv")
# df = pd.read_csv("./data/sintetis/dataset_mentah.csv")
print(f"Data: {df.shape[0]} baris, {df.shape[1]} kolom")
df.head()

Data: 3000 baris, 14 kolom


,air_temperature,air_humidity,soil_moisture,Reference evapotranspiration,Evapotranspiration,Crop Coefficient,Crop Coefficient stage,nitrogen,fosfor,kalium,Solar Radiation ghi,Wind Speed,plant_age,pH
0,31.2,93.6,567.0,563.000086,236.460036,0.42,Initial Stage,107,38,53,622.0,2.09,1,3.32
1,31.2,93.6,567.0,561.176578,235.694163,0.42,Initial Stage,107,38,53,622.0,2.09,1,3.77
2,30.5,74.6,307.0,561.267170,235.732211,0.42,Initial Stage,107,38,53,622.0,2.09,3,2.90
3,30.4,76.6,308.0,559.447778,234.968067,0.42,Initial Stage,107,38,53,622.0,2.09,3,7.71
4,30.4,76.6,308.0,559.447778,234.968067,0.42,Initial Stage,107,38,53,622.0,2.09,3,5.11


# Cek data

In [2]:
df[["soil_moisture", "air_temperature", "air_humidity"]].describe().round(2)

,soil_moisture,air_temperature,air_humidity
count,3000.00,3000.00,3000.00
mean,393.45,25.38,77.59
std,167.86,4.05,9.95
min,120.09,18.00,60.00
25%,240.28,21.96,69.18
50%,386.16,25.65,77.80
75%,539.57,28.90,85.80
max,699.99,32.00,98.20


## hapus data ga kepake dan normalisasi

In [3]:
# --- NORMALISASI SOIL MOISTURE KE PERSENTASE (0-100%) ---
sm_min = df_bersih["soil_moisture"].min()
sm_max = df_bersih["soil_moisture"].max()

# Rumus di balik (dikurang dari 100) asumsi nilai mentah tinggi = kering
df_bersih["soil_moisture"] = 100 - ((df_bersih["soil_moisture"] - sm_min) / (sm_max - sm_min) * 100)

# Kunci nilainya di rentang realistis 10% - 90%
df_bersih["soil_moisture"] = df_bersih["soil_moisture"].clip(10, 90).round(2)

print("Normalisasi beres!")
print(df_bersih[["soil_moisture"]].describe().round(2))

NameError: name 'df_bersih' is not defined

# Fungsi labelling

In [ ]:
# Acuan diambil dari petani:
#   - siram pagi  : soil_moisture ~11  (udara sejuk & lembap)
#   - siram jam 11: soil_moisture ~25  (udara panas 33C & kering 42%)

def label_irigasi(row):
    sm = row["soil_moisture"]
    at = row["air_temperature"]
    ah = row["air_humidity"]

    if sm <= 0 or sm > 45:      
        return -1

    threshold_kering = 12
    threshold_basah = 20

    if at > 32 and ah < 55:     
        threshold_kering += 13  
        threshold_basah += 10   
    elif ah > 85:               
        threshold_kering -= 1
    
    if sm < threshold_kering:
        action = 1              
    else:
        action = 0              

    return action

print("Fungsi label_irigasi() siap.")

Fungsi label_irigasi() siap.


# Terapkan labelling

In [ ]:
# Terapkan fungsi labeling
df_bersih["irrigation_action"] = df_bersih.apply(label_irigasi, axis=1)

# Filter baris yang action-nya -1 (data tidak masuk kriteria diabaikan)
df_final = df_bersih[df_bersih["irrigation_action"] != -1].reset_index(drop=True) 

print("Distribusi label:")
dist = df_final["irrigation_action"].value_counts().sort_index()
for val, count in dist.items():
    label = "Tidak siram" if val == 0 else "Siram"
    print(f"  {val} ({label}): {count} ({count/len(df_final)*100:.1f}%)")

Distribusi label:
  0 (Tidak siram): 827 (54.9%)
  1 (Siram): 680 (45.1%)


## Preview

In [ ]:
sample = df_final.head(20)
sample

,air_temperature,air_humidity,soil_moisture,Reference evapotranspiration,Evapotranspiration,Crop Coefficient,Crop Coefficient stage,Nitrogen [mg/kg],Phosphorus [mg/kg],Potassium,Solar Radiation ghi,Wind Speed,Days of planted,pH,irrigation_action
0,30.5,74.6,27.79,561.267170,235.732211,0.42,Initial Stage,107,38,53,622.0,2.09,3,2.90,0
1,30.4,76.6,28.00,559.447778,234.968067,0.42,Initial Stage,107,38,53,622.0,2.09,3,7.71,0
2,30.4,76.6,28.00,559.447778,234.968067,0.42,Initial Stage,107,38,53,622.0,2.09,3,5.11,0
3,30.3,77.1,27.79,558.984239,234.773380,0.42,Initial Stage,107,38,53,622.0,2.09,3,4.48,0
4,30.3,76.5,27.79,559.517077,234.997172,0.42,Initial Stage,107,38,53,622.0,2.09,3,4.32,0
5,30.3,78.1,27.79,558.100762,234.402320,0.42,Initial Stage,107,38,53,622.0,2.09,3,3.77,0
6,30.3,78.8,27.79,557.485689,234.143989,0.42,Initial Stage,107,38,53,622.0,2.09,3,3.68,0
7,30.3,78.8,27.79,557.485689,234.143989,0.42,Initial Stage,107,38,53,622.0,2.09,3,3.67,0
8,30.2,77.5,28.00,558.612080,234.617074,0.42,Initial Stage,107,38,53,622.0,2.09,3,3.95,0
9,30.2,77.6,27.79,558.523956,234.580061,0.42,Initial Stage,107,38,53,622.0,2.09,3,2.17,0


# save

# Simpan

In [ ]:
FEATURES = ["soil_moisture", "air_temperature", "air_humidity"]
cols = FEATURES + ["irrigation_action"]

df_final[cols].to_csv("data/dataset_irigasi.csv", index=False)
print("Tersimpan: data/dataset_irigasi.csv")
print(f"Total: {len(df)} baris, kolom: {cols}")


Tersimpan: data/dataset_irigasi.csv
Total: 3000 baris, kolom: ['soil_moisture', 'air_temperature', 'air_humidity', 'irrigation_action']
